# 03 — Modeling
### Credit Card Fraud Detection

This notebook:

1. Rebuilds the preprocessing pipeline from notebook 02 (self-contained —
   doesn't rely on notebook 02's kernel state)
2. Benchmarks all 5 model families × 2 imbalance strategies (class
   weighting vs. SMOTE) via 5-fold cross-validated PR-AUC — 10 combinations
3. Selects the top 3 combinations and tunes each with Optuna
4. Fits the overall champion on the full training set and saves it, ready
   for `04_evaluation.ipynb` to evaluate thoroughly on the untouched test set

**Scoping decision, stated upfront:** the `RobustScaler` preprocessing step
is fit once on the full training set (not refit inside every CV fold).
Scaling doesn't use label information, so this carries far less leakage
risk than resampling does — refitting a scaler per fold is best practice
but the marginal benefit here is small relative to the added complexity.
**SMOTE is the leakage-sensitive step and *is* refit inside every CV fold**,
via an `imblearn.pipeline.Pipeline`, for both the baseline comparison and
Optuna tuning.


In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
import joblib

from src.config import RANDOM_STATE, FIGURES_DIR, MODELS_DIR, PREPROCESSOR_FILE, MODEL_FILE
from src.data import load_processed_data
from src.features import add_engineered_features, build_preprocessor, get_scale_pos_weight
from src.models import (
    MODEL_REGISTRY,
    get_model,
    build_optuna_objective,
    save_model,
    save_training_summary,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)

%matplotlib inline


## 1. Rebuild the Preprocessing Pipeline

Reproduces notebook 02's pipeline exactly, so this notebook can be run
independently. Loads the deduplicated, split data that notebook 02 saved to
`data/processed/`.

In [ ]:
X_train, X_test, y_train, y_test = load_processed_data()
print(f"Train: {X_train.shape}, fraud rate: {y_train.mean() * 100:.4f}%")
print(f"Test:  {X_test.shape}, fraud rate: {y_test.mean() * 100:.4f}%")

X_train_fe = add_engineered_features(X_train)
X_test_fe = add_engineered_features(X_test)

preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)   # transform only, never fit

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

scale_pos_weight = get_scale_pos_weight(y_train)
print(f"\nX_train_processed shape: {X_train_processed.shape}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


## 2. Baseline Comparison: 5 Model Families × 2 Strategies

**Strategy A — class weighting:** each model's built-in imbalance handling
(`class_weight='balanced'` for logistic regression/random forest,
`scale_pos_weight` for the boosted-tree families). No resampling.

**Strategy B — SMOTE:** the model's built-in imbalance handling is
*disabled* (`class_weight=None` / `scale_pos_weight=1.0`), and SMOTE
oversamples the minority class to 10% of the majority count instead,
refit inside every CV fold.

Both are scored on **PR-AUC (average precision)** — the right metric here,
not accuracy or ROC-AUC, given the 0.17% positive rate.

⏱ This cell trains 10 combinations × 5 folds = 50 model fits. Expect this
to take several minutes — CatBoost/XGBoost at default settings are the
slowest of the five.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_results = []

for name in sorted(MODEL_REGISTRY):
    print(f"--- {name} ---")

    # Strategy A: class weighting, no resampling
    t0 = time.time()
    model_a = get_model(name, scale_pos_weight=scale_pos_weight, class_weight="balanced")
    cv_a = cross_validate(
        model_a, X_train_processed, y_train, cv=cv,
        scoring="average_precision", n_jobs=1,
    )
    elapsed_a = time.time() - t0
    baseline_results.append({
        "model": name, "strategy": "class_weight",
        "pr_auc_mean": cv_a["test_score"].mean(),
        "pr_auc_std": cv_a["test_score"].std(),
        "elapsed_sec": elapsed_a,
    })
    print(f"  class_weight: PR-AUC = {cv_a['test_score'].mean():.4f} "
          f"(+/- {cv_a['test_score'].std():.4f})  [{elapsed_a:.1f}s]")

    # Strategy B: SMOTE, model's own imbalance correction disabled
    t0 = time.time()
    model_b = get_model(name, scale_pos_weight=1.0, class_weight=None)
    pipeline_b = ImbPipeline([
        ("smote", SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)),
        ("clf", model_b),
    ])
    cv_b = cross_validate(
        pipeline_b, X_train_processed, y_train, cv=cv,
        scoring="average_precision", n_jobs=1,
    )
    elapsed_b = time.time() - t0
    baseline_results.append({
        "model": name, "strategy": "smote",
        "pr_auc_mean": cv_b["test_score"].mean(),
        "pr_auc_std": cv_b["test_score"].std(),
        "elapsed_sec": elapsed_b,
    })
    print(f"  smote:        PR-AUC = {cv_b['test_score'].mean():.4f} "
          f"(+/- {cv_b['test_score'].std():.4f})  [{elapsed_b:.1f}s]")

baseline_df = pd.DataFrame(baseline_results).sort_values("pr_auc_mean", ascending=False).reset_index(drop=True)
baseline_df


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

models_sorted = sorted(MODEL_REGISTRY)
x = np.arange(len(models_sorted))
width = 0.35

cw = baseline_df[baseline_df.strategy == "class_weight"].set_index("model").loc[models_sorted]
sm = baseline_df[baseline_df.strategy == "smote"].set_index("model").loc[models_sorted]

ax.bar(x - width / 2, cw["pr_auc_mean"], width, yerr=cw["pr_auc_std"],
       label="class_weight / scale_pos_weight", capsize=4)
ax.bar(x + width / 2, sm["pr_auc_mean"], width, yerr=sm["pr_auc_std"],
       label="SMOTE", capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(models_sorted, rotation=20)
ax.set_ylabel("PR-AUC (mean ± std, 5-fold CV)")
ax.set_title("Baseline comparison: imbalance strategy × model family")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "09_baseline_model_comparison.png", bbox_inches="tight")
plt.show()


**How to read this:** don't assume the more sophisticated technique
(SMOTE) wins by default — for tree-based models in particular,
`scale_pos_weight` is often just as effective while being far cheaper to
compute. Let the actual numbers above decide, not intuition.

## 3. Select Top 3 for Optuna Tuning

Tuning all 10 combinations at full trial counts would be expensive for
limited additional insight — the baseline comparison already tells us which
combinations are competitive. We focus tuning compute on the top 3 by mean
PR-AUC.

In [ ]:
TOP_N = 3
top_candidates = baseline_df.head(TOP_N)[["model", "strategy", "pr_auc_mean"]].to_dict("records")

print(f"Top {TOP_N} candidates selected for Optuna tuning:")
for c in top_candidates:
    print(f"  {c['model']:20s} | {c['strategy']:12s} | baseline PR-AUC = {c['pr_auc_mean']:.4f}")


## 4. Optuna Tuning

Each candidate gets its own study, optimizing mean PR-AUC across
`StratifiedKFold` CV. For candidates using the `smote` strategy, SMOTE is
refit inside every fold of every trial (via the `use_smote=True` flag added
to `build_optuna_objective` in Phase 7's `src/models.py` update) — same
leakage-safe approach as the baseline comparison above.

Using 3-fold CV (rather than 5) and a moderate trial count during search
to keep runtime reasonable — increase `N_TRIALS` for a more thorough search
once you've confirmed everything runs end-to-end.

In [ ]:
N_TRIALS = 30
N_SPLITS_TUNING = 3

tuning_results = {}

for candidate in top_candidates:
    name, strategy = candidate["model"], candidate["strategy"]
    use_smote = strategy == "smote"
    key = f"{name}__{strategy}"

    print(f"\n=== Tuning {name} ({strategy}) — {N_TRIALS} trials ===")
    t0 = time.time()

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    objective = build_optuna_objective(
        name, X_train_processed, y_train,
        n_splits=N_SPLITS_TUNING, use_smote=use_smote,
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    elapsed = time.time() - t0
    tuning_results[key] = {
        "model": name,
        "strategy": strategy,
        "baseline_pr_auc": candidate["pr_auc_mean"],
        "tuned_pr_auc": study.best_value,
        "best_params": study.best_params,
        "study": study,
        "elapsed_sec": elapsed,
    }

    print(f"Baseline PR-AUC: {candidate['pr_auc_mean']:.4f}  ->  "
          f"Tuned PR-AUC: {study.best_value:.4f}  "
          f"({'+' if study.best_value >= candidate['pr_auc_mean'] else ''}"
          f"{(study.best_value - candidate['pr_auc_mean']):.4f})  [{elapsed / 60:.1f} min]")


In [ ]:
tuning_summary_df = pd.DataFrame([
    {
        "model": v["model"],
        "strategy": v["strategy"],
        "baseline_pr_auc": v["baseline_pr_auc"],
        "tuned_pr_auc": v["tuned_pr_auc"],
        "improvement": v["tuned_pr_auc"] - v["baseline_pr_auc"],
    }
    for v in tuning_results.values()
]).sort_values("tuned_pr_auc", ascending=False).reset_index(drop=True)

tuning_summary_df


## 5. Champion Selection

The candidate with the highest **tuned** CV PR-AUC becomes the champion carried into evaluation (Phase 8).

In [ ]:
champion_key = tuning_summary_df.iloc[0][["model", "strategy"]].pipe(lambda r: f"{r['model']}__{r['strategy']}")
champion = tuning_results[champion_key]

print(f"🏆 Champion: {champion['model']} ({champion['strategy']})")
print(f"   Tuned CV PR-AUC: {champion['tuned_pr_auc']:.4f}")
print(f"   Best params: {json.dumps(champion['best_params'], indent=2)}")


In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

fig1 = plot_optimization_history(champion["study"])
fig1.set_title(f"Optimization history — {champion['model']} ({champion['strategy']})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "10_optuna_optimization_history.png", bbox_inches="tight")
plt.show()

fig2 = plot_param_importances(champion["study"])
fig2.set_title(f"Hyperparameter importance — {champion['model']} ({champion['strategy']})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "11_optuna_param_importances.png", bbox_inches="tight")
plt.show()


## 6. Fit the Champion on the Full Training Set

This is a single final fit (no CV) using the champion's tuned
hyperparameters, on the *entire* training set. If the champion's strategy
is `smote`, we apply SMOTE once here — this is safe because there's no
held-out fold being evaluated within this step; the true held-out set
(`X_test_processed`) is completely untouched and reserved for
`04_evaluation.ipynb`.

In [ ]:
from src.features import apply_smote

use_smote_final = champion["strategy"] == "smote"

if use_smote_final:
    X_train_final, y_train_final = apply_smote(
        pd.DataFrame(X_train_processed), y_train, sampling_strategy=0.1
    )
    X_train_final = X_train_final.values
    print(f"Fitting on SMOTE-resampled data: {X_train_final.shape}, "
          f"fraud rate: {y_train_final.mean() * 100:.2f}%")
else:
    X_train_final, y_train_final = X_train_processed, y_train
    print(f"Fitting on original (class-weighted) data: {X_train_final.shape}")

final_model = get_model(
    champion["model"],
    params=champion["best_params"],
    scale_pos_weight=(1.0 if use_smote_final else scale_pos_weight),
    class_weight=(None if use_smote_final else "balanced"),
)
final_model.fit(X_train_final, y_train_final)
print("✅ Champion model fit on full training set")


## 7. Save Artifacts

Saves the fitted preprocessor, the champion model, and a training summary
recording which family/strategy/hyperparameters won — everything
`04_evaluation.ipynb` and eventually `api/` (Phase 9) need.

In [ ]:
joblib.dump(preprocessor, PREPROCESSOR_FILE)
print(f"✅ Preprocessor saved to {PREPROCESSOR_FILE}")

save_model(final_model, path=MODEL_FILE)

save_training_summary(
    model_family=champion["model"],
    strategy=champion["strategy"],
    best_params=champion["best_params"],
    cv_pr_auc=champion["tuned_pr_auc"],
    model=final_model,
)


## 8. Summary

Fill in after running:

1. **Baseline winner (pre-tuning):** lightgbm , smote, 0.8499
2. **Did SMOTE beat class weighting, or the reverse?** smote was the winner across all the models 
3. **Champion after tuning:** lightgbm, smote, PR-AUC: 0.8626, (+0.0127)
4. **Most important hyperparameters for the champion** 🏆 Champion: lightgbm (smote)
   Tuned CV PR-AUC: 0.8626
   Best params: {
  "n_estimators": 445,
  "num_leaves": 107,
  "learning_rate": 0.12280434615435543,
  "subsample": 0.6507854148078054,
  "colsample_bytree": 0.9045329551721945,
  "min_child_samples": 37,
  "reg_lambda": 0.023816324846407885
}
5. **Next step:** `04_evaluation.ipynb` loads model.cbm 
   `.joblib`, per `training_summary.json`) and `models/preprocessor.joblib`,
   evaluates on the untouched `X_test_processed`/`y_test`, tunes the
   decision threshold, and runs SHAP for interpretability.
